# Create z.npy from X.npy

This notebook creates `z.npy` which indicates whose turn it is:
- `1` = Player 1's turn
- `-1` = Player 2's turn

The turn is determined by counting the total pieces on the board:
- If total pieces is even → Player 1's turn (they go first)
- If total pieces is odd → Player 2's turn

In [1]:
import numpy as np

In [2]:
# Load the existing data
X = np.load('Dataset/X.npy')
y = np.load('Dataset/y.npy')

print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}, dtype: {y.dtype}")

X shape: (92559, 6, 7, 2), dtype: int8
y shape: (92559,), dtype: int64


In [3]:
# X has shape (N, 6, 7, 2) where:
# - Channel 0: Player 1 pieces (1 where player 1 has a piece)
# - Channel 1: Player 2 pieces (1 where player 2 has a piece)

# Count total pieces for each board
# Sum across all spatial dimensions (6, 7) and both channels (2)
total_pieces = X.sum(axis=(1, 2, 3))

print(f"Total pieces shape: {total_pieces.shape}")
print(f"Total pieces range: [{total_pieces.min()}, {total_pieces.max()}]")
print(f"Sample total pieces (first 10): {total_pieces[:10]}")

Total pieces shape: (92559,)
Total pieces range: [0, 41]
Sample total pieces (first 10): [0 1 2 3 4 5 6 7 8 9]


In [4]:
# Determine whose turn it is:
# - Even total pieces → Player 1's turn (z = 1)
# - Odd total pieces → Player 2's turn (z = -1)

z = np.where(total_pieces % 2 == 0, 1, -1).astype(np.int64)

print(f"z shape: {z.shape}, dtype: {z.dtype}")
print(f"z unique values: {np.unique(z)}")
print(f"z distribution: Player 1's turn: {(z == 1).sum()}, Player 2's turn: {(z == -1).sum()}")

z shape: (92559,), dtype: int64
z unique values: [-1  1]
z distribution: Player 1's turn: 47081, Player 2's turn: 45478


In [5]:
# Save z.npy
np.save('Dataset/z.npy', z)
print("Saved: Dataset/z.npy")

Saved: Dataset/z.npy


In [6]:
# Verify by loading back
z_loaded = np.load('Dataset/z.npy')
print(f"Verification:")
print(f"  z shape: {z_loaded.shape}")
print(f"  z dtype: {z_loaded.dtype}")
print(f"  z unique values: {np.unique(z_loaded)}")
print(f"  Arrays equal: {np.array_equal(z, z_loaded)}")

Verification:
  z shape: (92559,)
  z dtype: int64
  z unique values: [-1  1]
  Arrays equal: True


In [7]:
# Optional: Verify a few samples visually
print("Sample verification:")
for i in range(3):
    board = X[i]
    p1_pieces = board[..., 0].sum()
    p2_pieces = board[..., 1].sum()
    total = p1_pieces + p2_pieces
    turn = z[i]
    print(f"Sample {i}: P1 pieces={p1_pieces}, P2 pieces={p2_pieces}, Total={total}, Turn={turn} ({'P1' if turn == 1 else 'P2'})")

Sample verification:
Sample 0: P1 pieces=0, P2 pieces=0, Total=0, Turn=1 (P1)
Sample 1: P1 pieces=0, P2 pieces=1, Total=1, Turn=-1 (P2)
Sample 2: P1 pieces=1, P2 pieces=1, Total=2, Turn=1 (P1)
